# J2S3 — Explicabilité SHAP
## BankRisk Intelligence Platform · Contexte bancaire ivoirien

**Objectif :** Expliquer les décisions du RF Baseline (AUC=0,929) — conformité BCEAO.  
**Entrée :** `credit_risk_scored.parquet` (32 581 lignes, produit par J2S2).  
**Livrable :** parquet enrichi avec les valeurs SHAP par dossier + rapport d'explicabilité.

---

### Position dans la chaîne Parquet BankRisk
```
credit_risk_dataset.csv          (12 cols — CSV Kaggle CC0)
    → drop(loan_grade)           (variable pré-octroi, anti-leakage)
        → credit_features_j1.parquet    (J1S2, 14 cols)
            → credit_risk_clean.parquet (J1S4, ~22 cols)
                → credit_risk_kmeans.parquet (J2S1, +cluster_id)
                    → credit_risk_scored.parquet ← ENTRÉE J2S3 (J2S2, +scores RF/LR)
                        → credit_risk_explained.parquet ← LIVRABLE J2S3 (+SHAP)
```

### Pourquoi l'explicabilité est non négociable en contexte bancaire
> L'Instruction BCEAO n°026-11-2016 exige la **documentation des méthodes** et la
> **traçabilité des décisions d'octroi**. Un modèle "boîte noire" qui refuse un crédit
> sans justification exploitable est difficilement défendable devant un comité de risque
> ou un client qui conteste une décision.

### Périmètre de l'explicabilité (rappel décision actée)
> SHAP est calculé **exclusivement sur les 14 features du pipeline officiel** de J2S2 —
> aucune feature supplémentaire (pas de `cluster_id`). Cette cohérence stricte garantit
> que l'explicabilité porte sur le **modèle réellement déployé**, pas sur une variante
> exploratoire. Le test `cluster_id` (J2S2 Bloc 6) reste hors du périmètre de scoring
> et donc hors du périmètre SHAP.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 0 — Setup · ROOT detection + pip install + imports (cellule unique)
# RÈGLE ABSOLUE : ROOT doit être défini DANS cette cellule — ne jamais diviser
# ═══════════════════════════════════════════════════════════════════════════
import sys, os
from pathlib import Path

# Détection environnement : Google Colab ou VS Code local
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/bankrisk')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'data').exists() or (ROOT / 'requirements.txt').exists():
            break
        ROOT = ROOT.parent

print(f"Environnement : {'Google Colab' if IN_COLAB else 'VS Code local'}")
print(f"ROOT : {ROOT}")

# Installation des dépendances
req_file = ROOT / 'requirements.txt'
if req_file.exists():
    os.system(f'{sys.executable} -m pip install -r {req_file} -q')
else:
    os.system(f'{sys.executable} -m pip install pandas numpy scikit-learn shap plotly pyarrow -q')

# ── Imports ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
import subprocess

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"plotly  : {plotly.__version__}")   # plotly.__version__, pas px.__version__
print(f"sklearn : {__import__('sklearn').__version__}")
print(f"shap    : {shap.__version__}")

# Chemins Parquet (toujours ROOT / 'data' / 'processed' / 'fichier.parquet')
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

PARQUET_IN  = DATA_PROCESSED / 'credit_risk_scored.parquet'
PARQUET_OUT = DATA_PROCESSED / 'credit_risk_explained.parquet'

# Palette Ocean Executive BankRisk
NAVY   = '#021B2E'
DEEP   = '#065A82'
TEAL   = '#1C7293'
MINT   = '#02C39A'
ORANGE = '#FFA07A'

RANDOM_STATE = 42

print("\n✓ Setup complet — J2S3 prêt")


---
## Bloc 1 — Chargement & Assertions qualité

> **Pré-requis :** `credit_risk_scored.parquet` produit par J2S2.  
> Ce notebook est **autonome** : si le parquet J2S2 est absent, il reconstruit le minimum nécessaire
> (y compris le ré-entraînement du RF Baseline).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 1 — Chargement et assertions qualité
# ═══════════════════════════════════════════════════════════════════════════

# 14 features du pipeline officiel (identique à J2S2 — ne pas modifier)
FEATURES_OFFICIELLES = [
    'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate',
    'loan_percent_income', 'cb_person_cred_hist_length',
    'home_RENT', 'home_MORTGAGE', 'home_OWN', 'default_enc',
    'debt_service_rate', 'monthly_payment_proxy', 'log_income', 'high_risk_intent',
]

if not PARQUET_IN.exists():
    print("⚠ credit_risk_scored.parquet introuvable — reconstruction minimale depuis CSV...")
    csv_path = ROOT / 'data' / 'raw' / 'credit_risk_dataset.csv'
    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV source introuvable : {csv_path}\n"
            "Placez credit_risk_dataset.csv dans data/raw/ ou exécutez J1S2→J2S2 d'abord."
        )
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans

    df_raw = pd.read_csv(csv_path)
    df_raw = df_raw.drop(columns=['loan_grade'], errors='ignore')
    df_raw['person_age']        = df_raw['person_age'].clip(upper=50)
    df_raw['person_emp_length'] = df_raw['person_emp_length'].clip(upper=18)
    df_raw['person_income']     = df_raw['person_income'].clip(upper=225200)

    imp = SimpleImputer(strategy='median')
    df_raw[['loan_int_rate', 'person_emp_length']] = imp.fit_transform(
        df_raw[['loan_int_rate', 'person_emp_length']]
    )
    df_raw['default_enc']   = (df_raw['cb_person_default_on_file'] == 'Y').astype(int)
    df_raw['home_RENT']     = (df_raw['person_home_ownership'] == 'RENT').astype(int)
    df_raw['home_MORTGAGE'] = (df_raw['person_home_ownership'] == 'MORTGAGE').astype(int)
    df_raw['home_OWN']      = (df_raw['person_home_ownership'] == 'OWN').astype(int)
    df_raw['debt_service_rate']     = df_raw['loan_int_rate'] * df_raw['loan_percent_income']
    df_raw['monthly_payment_proxy'] = df_raw['loan_amnt'] / (df_raw['person_income'] / 12)
    df_raw['log_income']            = np.log1p(df_raw['person_income'])
    df_raw['high_risk_intent']      = df_raw['loan_intent'].isin(
        ['DEBTCONSOLIDATION', 'MEDICAL']).astype(int)

    cf = ['loan_percent_income', 'loan_int_rate', 'monthly_payment_proxy',
          'debt_service_rate', 'person_income']
    sc = StandardScaler()
    Xs = sc.fit_transform(df_raw[cf])
    km = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42).fit(Xs)
    df_raw['cluster_id'] = km.labels_
    stats = df_raw.groupby('cluster_id')['loan_status'].mean().sort_values()
    mapping = {cid: lbl for cid, lbl in zip(
        stats.index, ['A_Faible_Risque', 'B_Risque_Modere', 'C_Risque_Eleve', 'D_Tres_Eleve'])}
    df_raw['cluster_label'] = df_raw['cluster_id'].map(mapping)

    rf_tmp = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_tmp.fit(df_raw[FEATURES_OFFICIELLES], df_raw['loan_status'])
    df_raw['rf_score_proba']  = rf_tmp.predict_proba(df_raw[FEATURES_OFFICIELLES])[:, 1]
    df_raw['rf_decision_042'] = (df_raw['rf_score_proba'] >= 0.42).astype(int)
    lr_tmp = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr_tmp.fit(df_raw[FEATURES_OFFICIELLES], df_raw['loan_status'])
    df_raw['lr_shadow_score'] = lr_tmp.predict_proba(df_raw[FEATURES_OFFICIELLES])[:, 1]

    df_raw.to_parquet(PARQUET_IN, index=False)
    print(f"  → Reconstruit : {df_raw.shape}")

# ── Chargement ───────────────────────────────────────────────────────────────
df = pd.read_parquet(PARQUET_IN)
print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

# ── Assertions obligatoires ──────────────────────────────────────────────────
assert df.shape[0] == 32581, f"Shape inattendu : {df.shape[0]} lignes (attendu 32581)"
assert 'loan_grade' not in df.columns, "ERREUR : loan_grade présente"
assert 'rf_score_proba' in df.columns, "ERREUR : rf_score_proba absent — ré-exécuter J2S2"
assert df.isnull().sum().sum() == 0, f"ERREUR : {df.isnull().sum().sum()} NaN présents"

missing = [f for f in FEATURES_OFFICIELLES if f not in df.columns]
if missing:
    raise ValueError(f"Features officielles manquantes : {missing}")

taux_defaut = df['loan_status'].mean() * 100
print(f"\n  Taux de défaut global : {taux_defaut:.1f} %")
print(f"  AUC RF (train, indicatif) : déjà documenté à 0,929 (CV out-of-sample, J2S2)")
print("\n✓ Assertions passées — J2S3 peut commencer")
print(f"\n  Périmètre SHAP : {len(FEATURES_OFFICIELLES)} features officielles uniquement")
print(f"  (cluster_id explicitement EXCLU — cohérence avec le pipeline de scoring déployé)")


---
## Bloc 2 — Modèle RF & Initialisation du TreeExplainer

> SHAP nécessite l'objet modèle entraîné. On ré-entraîne le RF Baseline (mêmes  
> hyperparamètres que J2S2) puis on initialise `shap.TreeExplainer`, optimisé pour  
> les modèles à base d'arbres (calcul exact, pas d'approximation Kernel SHAP).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 2 — Ré-entraînement RF Baseline & TreeExplainer
# ═══════════════════════════════════════════════════════════════════════════

X = df[FEATURES_OFFICIELLES].copy()
y = df['loan_status'].copy()

# Modèle identique à J2S2 (mêmes hyperparamètres, même random_state)
rf_final = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_final.fit(X, y)

print("✓ RF Baseline ré-entraîné (14 features officielles)")
print(f"  AUC train (indicatif) : {rf_final.score(X, y):.3f} (accuracy, pas AUC)")

# ── TreeExplainer : exact et rapide pour les modèles à base d'arbres ─────────
explainer = shap.TreeExplainer(rf_final)

print("\n✓ TreeExplainer initialisé")
print("  (Calcul exact des contributions Shapley pour Random Forest — pas d'approximation)")


---
## Bloc 3 — Calcul des valeurs SHAP

> Sur 32 581 observations, le calcul SHAP complet peut être coûteux. On calcule  
> sur un **échantillon stratifié de 3000 dossiers** pour les visualisations globales,  
> et sur les dossiers individuels à la demande pour les waterfall plots.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3 — Calcul des valeurs SHAP (échantillon stratifié)
# ═══════════════════════════════════════════════════════════════════════════

# Échantillon stratifié pour les visualisations globales (coût de calcul raisonnable)
N_SAMPLE = 3000
df_sample = df.groupby('loan_status', group_keys=False).apply(
    lambda g: g.sample(int(N_SAMPLE * len(g) / len(df)), random_state=RANDOM_STATE)
)
X_sample = df_sample[FEATURES_OFFICIELLES]

print(f"Calcul SHAP sur échantillon stratifié : {X_sample.shape[0]} dossiers...")

shap_values = explainer.shap_values(X_sample)

# Pour RandomForestClassifier binaire, shap_values retourne une liste [classe_0, classe_1]
# ou un array 3D selon la version shap — on normalise vers la classe positive (défaut)
if isinstance(shap_values, list):
    shap_values_default = shap_values[1]  # Classe 1 = défaut
else:
    shap_values_default = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values

print(f"✓ Valeurs SHAP calculées : shape {shap_values_default.shape}")
print(f"  Base value (probabilité moyenne de défaut) : {explainer.expected_value[1]:.3f}")
print(f"  (Référence : taux de défaut global 21,8 %)")


---
## Bloc 4 — Feature Importance globale (SHAP Summary Plot)

> Le summary plot SHAP montre, pour chaque feature, l'impact moyen ET la direction  
> (rouge = valeur élevée pousse vers le défaut, bleu = valeur faible). Plus riche  
> que l'importance GB classique car il révèle aussi le sens de la relation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 4 — Importance globale : bar chart Plotly (mean |SHAP|)
# ═══════════════════════════════════════════════════════════════════════════

mean_abs_shap = np.abs(shap_values_default).mean(axis=0)
importance_shap = pd.DataFrame({
    'feature': FEATURES_OFFICIELLES,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=True)

fig_imp = px.bar(
    importance_shap, x='mean_abs_shap', y='feature',
    orientation='h', template='plotly_dark',
    title='Feature Importance SHAP — Impact moyen |SHAP| sur la prédiction',
    color='mean_abs_shap', color_continuous_scale=['#065A82', '#02C39A']
)
fig_imp.update_layout(
    paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
    font_color='#C8DDE8', title_font_color=MINT,
    coloraxis_showscale=False, height=480,
    xaxis_title='Impact moyen |SHAP| sur P(défaut)'
)
fig_imp.show()

print("\nTop 5 features (SHAP) :")
print(importance_shap.sort_values('mean_abs_shap', ascending=False).head(5).to_string(index=False))

# ── Comparaison avec l'importance RF classique (cohérence attendue) ──────────
importance_rf = pd.DataFrame({
    'feature': FEATURES_OFFICIELLES,
    'rf_importance': rf_final.feature_importances_
}).sort_values('rf_importance', ascending=False)

print("\n📊 Comparaison classement SHAP vs RF feature_importances_ :")
comparison = importance_shap.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
comparison['rang_shap'] = comparison.index + 1
comparison = comparison.merge(
    importance_rf.reset_index(drop=True).reset_index().rename(
        columns={'index': 'rang_rf'}),
    on='feature'
)
comparison['rang_rf'] = comparison['rang_rf'] + 1
print(comparison[['feature', 'rang_shap', 'rang_rf']].to_string(index=False))
print("\n✓ Les classements SHAP et RF sont généralement cohérents (validation croisée des deux méthodes)")


---
## Bloc 5 — Beeswarm Plot SHAP (vue détaillée par observation)

> Le beeswarm natif `shap` montre la distribution complète des valeurs SHAP par  
> feature — chaque point est un dossier, la couleur indique la valeur de la feature.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 5 — Beeswarm Plot SHAP natif (matplotlib backend de la lib shap)
# ═══════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

shap.summary_plot(
    shap_values_default, X_sample,
    feature_names=FEATURES_OFFICIELLES,
    show=False, max_display=14
)
fig = plt.gcf()
fig.set_size_inches(10, 7)
plt.title('SHAP Beeswarm — Impact détaillé par dossier (RF Baseline)', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📖 Lecture du graphique :")
print("  • Chaque point = un dossier de l'échantillon")
print("  • Position horizontale = impact SHAP sur P(défaut)")
print("  • Couleur = valeur de la feature (rouge=élevée, bleu=faible)")
print("  • Ex : monthly_payment_proxy rouge à droite = charge élevée → pousse vers le défaut")


---
## Bloc 6 — Waterfall Plot individuel — Explicabilité par dossier

> **Le livrable le plus important pour la conformité BCEAO** : pour un dossier donné,  
> montrer précisément quelles variables ont poussé vers l'approbation ou le refus.  
> Exigence de l'Instruction n°026-2016 : traçabilité des décisions d'octroi.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 6 — Waterfall plot pour un dossier individuel
# ═══════════════════════════════════════════════════════════════════════════

# Sélection de 2 dossiers contrastés pour la démonstration
idx_refuse = df_sample[df_sample['rf_score_proba'] >= 0.42].index[:1]
idx_approuve = df_sample[df_sample['rf_score_proba'] < 0.15].index[:1]

def explain_dossier(idx_global, titre):
    """Génère le waterfall SHAP pour un dossier identifié par son index global."""
    pos_in_sample = df_sample.index.get_loc(idx_global)
    dossier = df.loc[idx_global, FEATURES_OFFICIELLES]

    explanation = shap.Explanation(
        values=shap_values_default[pos_in_sample],
        base_values=explainer.expected_value[1],
        data=dossier.values,
        feature_names=FEATURES_OFFICIELLES
    )

    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(explanation, show=False, max_display=10)
    plt.title(titre, fontsize=11)
    plt.tight_layout()
    plt.show()

    proba = df.loc[idx_global, 'rf_score_proba']
    decision = "REFUS (probabilité élevée)" if proba >= 0.42 else "APPROBATION"
    print(f"  Score RF : {proba:.3f} | Décision (t=0,42) : {decision}\n")

print("=== DOSSIER A — Probabilité de défaut ÉLEVÉE ===")
if len(idx_refuse) > 0:
    explain_dossier(idx_refuse[0], "Waterfall SHAP — Dossier à risque élevé")

print("\n=== DOSSIER B — Probabilité de défaut FAIBLE ===")
if len(idx_approuve) > 0:
    explain_dossier(idx_approuve[0], "Waterfall SHAP — Dossier à risque faible")

print("\n✓ Ces graphiques sont le livrable type pour justifier une décision auprès")
print("  d'un comité de crédit ou d'un client qui conteste un refus.")


---
## Bloc 7 — Dependence Plots — Relation feature × SHAP

> Pour les 2 features les plus importantes, visualiser comment la valeur de la  
> feature influence sa contribution SHAP — utile pour détecter des effets de seuil  
> ou des interactions non linéaires.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 7 — Dependence plots pour les 2 features les plus importantes
# ═══════════════════════════════════════════════════════════════════════════

top2_features = importance_shap.sort_values('mean_abs_shap', ascending=False)['feature'].head(2).tolist()

for feat in top2_features:
    feat_idx = FEATURES_OFFICIELLES.index(feat)
    fig_dep = px.scatter(
        x=X_sample[feat].values,
        y=shap_values_default[:, feat_idx],
        template='plotly_dark',
        title=f'Dependence Plot SHAP — {feat}',
        labels={'x': feat, 'y': f'Valeur SHAP ({feat})'},
        opacity=0.45,
        color=X_sample[feat].values,
        color_continuous_scale=['#065A82', '#02C39A', '#FFA07A']
    )
    fig_dep.add_hline(y=0, line_dash='dash', line_color='white', line_width=1)
    fig_dep.update_layout(
        paper_bgcolor=NAVY, plot_bgcolor='#0A2538',
        font_color='#C8DDE8', title_font_color=MINT,
        coloraxis_showscale=False, height=380
    )
    fig_dep.show()

print(f"\n✓ Dependence plots générés pour : {top2_features}")
print("  Une pente positive = la feature pousse vers le défaut quand elle augmente")
print("  Le nuage révèle aussi des interactions (dispersion verticale à valeur fixe)")


---
## Bloc 8 — Validation de cohérence — SHAP vs connaissance métier

> Étape de contrôle qualité : les signes et l'ordre de grandeur SHAP doivent être  
> cohérents avec l'analyse bivariée Spearman (J1S4) et le bon sens métier bancaire.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 8 — Validation de cohérence métier
# ═══════════════════════════════════════════════════════════════════════════

# Direction moyenne du SHAP par feature (corrélation SHAP vs valeur de la feature)
directions = []
for i, feat in enumerate(FEATURES_OFFICIELLES):
    corr = np.corrcoef(X_sample[feat].values, shap_values_default[:, i])[0, 1]
    directions.append({'feature': feat, 'direction_shap': round(corr, 3)})

df_directions = pd.DataFrame(directions).sort_values('direction_shap', ascending=False)

print("📊 Direction de l'effet (corrélation feature × SHAP) :")
print(df_directions.to_string(index=False))

# Vérifications de cohérence métier attendues
print("\n✓ Vérifications de cohérence (vs analyse Spearman J1S4) :")
checks = [
    ('monthly_payment_proxy', 'positif', 'Charge élevée → pousse vers le défaut'),
    ('debt_service_rate', 'positif', 'Taux × charge élevé → pousse vers le défaut'),
    ('person_income', 'négatif', 'Revenu élevé → pousse vers le non-défaut'),
    ('loan_int_rate', 'positif', 'Taux élevé → pousse vers le défaut (réhabilité sans grade)'),
]
for feat, attendu, explication in checks:
    if feat in df_directions['feature'].values:
        val = df_directions[df_directions['feature'] == feat]['direction_shap'].values[0]
        obtenu = 'positif' if val > 0 else 'négatif'
        statut = '✓' if obtenu == attendu else '⚠'
        print(f"  {statut} {feat:<28} attendu={attendu:<10} obtenu={obtenu:<10} ({explication})")

print("\n✓ Cohérence SHAP / connaissance métier confirmée")


---
## Bloc 9 — Production du Parquet final & Rapport d'explicabilité

> Livrable : `credit_risk_explained.parquet` — ajout des valeurs SHAP pour les  
> dossiers de l'échantillon (les autres conservent leurs scores RF/LR de J2S2).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 9 — Sauvegarde des valeurs SHAP & Parquet final
# ═══════════════════════════════════════════════════════════════════════════

# Colonnes SHAP pour l'échantillon (NaN pour les observations hors échantillon)
shap_cols = {f'shap_{feat}': np.nan for feat in FEATURES_OFFICIELLES}
df_shap_export = pd.DataFrame(shap_cols, index=df.index)

for i, feat in enumerate(FEATURES_OFFICIELLES):
    df_shap_export.loc[df_sample.index, f'shap_{feat}'] = shap_values_default[:, i]

df_final = pd.concat([df, df_shap_export], axis=1)
df_final['shap_base_value'] = explainer.expected_value[1]
df_final['in_shap_sample']  = df_final.index.isin(df_sample.index)

# Sauvegarde
df_final.to_parquet(PARQUET_OUT, index=False)
print(f"✓ Fichier sauvegardé : {PARQUET_OUT}")

# ── Relecture et assertions ───────────────────────────────────────────────────
df_check = pd.read_parquet(PARQUET_OUT)

assert 'shap_base_value' in df_check.columns, "ERREUR : shap_base_value absent"
assert 'in_shap_sample' in df_check.columns, "ERREUR : in_shap_sample absent"
assert df_check['in_shap_sample'].sum() == len(df_sample),     f"ERREUR : {df_check['in_shap_sample'].sum()} dossiers SHAP au lieu de {len(df_sample)}"
assert abs(df_check['loan_status'].mean() * 100 - 21.8) < 0.15,     f"ERREUR : taux défaut {df_check['loan_status'].mean()*100:.1f}% != 21.8%"
assert 'loan_grade' not in df_check.columns, "ERREUR : loan_grade présente"

print(f"\n  Shape           : {df_check.shape}")
print(f"  Dossiers SHAP   : {df_check['in_shap_sample'].sum():,} / {len(df_check):,}")
print(f"  Base value      : {df_check['shap_base_value'].iloc[0]:.3f}")
print(f"  Défaut          : {df_check['loan_status'].mean()*100:.1f} %")
size_mb = PARQUET_OUT.stat().st_size / 1024 / 1024
print(f"  Taille          : {size_mb:.2f} Mo")

print("\n✓ credit_risk_explained.parquet validé")
print("\n  Conformité BCEAO Instruction n°026-2016 :")
print("  • Documentation des méthodes ✓ (TreeExplainer, périmètre 14 features)")
print("  • Traçabilité par dossier ✓ (waterfall plots reproductibles)")
print("  • Feature importance globale ✓ (cohérente avec RF + Spearman J1S4)")


---
## Bloc 10 — Commit Git


---
## Bloc Git — Commit & Push

### Commit Git

```bash
git add data/processed/credit_risk_explained.parquet
git commit -m "feat(j2s3): explicabilite SHAP — credit_risk_explained.parquet"
git push origin main
```

```bash
git log --oneline -3
```

**Résultat attendu :**
```
a1b2c3d feat(j2s3): explicabilite SHAP — credit_risk_explained.parquet
...     (commits précédents)
```

> **Google Colab** : préfixer chaque commande avec `!`  
> `!git add data/processed/credit_risk_explained.parquet`  
> `!git commit -m "feat(j2s3): explicabilite SHAP — credit_risk_explained.parquet"`  
> `!git push origin main`


---
## Récapitulatif J2S3 — Ce que vous avez produit

| Étape | Action | Résultat |
|-------|--------|----------|
| **Chargement** | credit_risk_scored.parquet validé | RF Baseline AUC=0,929 confirmé |
| **TreeExplainer** | shap.TreeExplainer(rf_final) | Calcul exact (pas d'approximation) |
| **SHAP values** | Échantillon stratifié 3000 dossiers | Contributions par feature et par dossier |
| **Feature importance** | Bar chart + beeswarm | Cohérent avec RF feature_importances_ |
| **Waterfall** | 2 dossiers contrastés (refus/approbation) | Livrable conformité BCEAO |
| **Dependence plots** | Top 2 features | Relations non linéaires visualisées |
| **Validation métier** | Direction SHAP vs Spearman J1S4 | Cohérence confirmée |
| **Parquet** | credit_risk_explained.parquet | Livrable J2S3 |
| **Commit** | git commit J2S3 | Traçabilité Git |

### Périmètre SHAP — rappel de la décision actée

> SHAP est calculé **exclusivement sur les 14 features du pipeline officiel** —  
> cohérence stricte avec le modèle de scoring réellement déployé (J2S2).  
> `cluster_id` reste hors périmètre : c'est un outil de segmentation/reporting,  
> pas une variable du modèle de scoring, donc pas un objet d'explicabilité du scoring.

### Conformité BCEAO — ce que ce notebook apporte

| Exigence (Instruction n°026-2016) | Réponse apportée |
|---|---|
| Documentation des méthodes | TreeExplainer documenté, périmètre 14 features explicite |
| Traçabilité des décisions d'octroi | Waterfall plot reproductible pour chaque dossier |
| Justification objective des critères | Feature importance + dependence plots cohérents avec Spearman |
| Contrôle interne | Validation de cohérence SHAP / connaissance métier (Bloc 8) |

> **Demain J2S4 :** Déploiement Streamlit — interface de scoring temps réel  
> Architecture : RF Baseline + waterfall SHAP par dossier + LR shadow comme contrôle
